In [0]:
# Cài đặt thư viện websocket và kafka client. 
%pip install websocket-client confluent-kafka

In [0]:
dbutils.library.restartPython()

In [0]:
# Cấu hình Confluent Kafka
KAFKA_SERVER = "pkc-ldvr1.asia-southeast1.gcp.confluent.cloud:9092"
KAFKA_API_KEY = "LXF7KCAF7HH4LG4C"
KAFKA_API_SECRET = "cfltN30DvSA1L9BIEYAhkI/HelSSGX96AwRFG5npmt9Ef4Ihkv/uEYOcbQbQOZtw"

In [0]:
import json
import time
import logging
import websocket
from confluent_kafka import Producer

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

class DatabricksBinanceProducer:
    def __init__(self, symbol="btcusdt", interval="1m"):
        self.symbol = symbol
        self.ws_url = f"wss://fs-stream.binance.com/ws/{symbol}@kline_{interval}"
        self.topic = KAFKA_TOPIC
        
        # Cấu hình bảo mật kết nối tới Confluent Cloud
        self.producer = Producer({
            'bootstrap.servers': KAFKA_SERVER,
            'security.protocol': 'SASL_SSL',
            'sasl.mechanisms': 'PLAIN',
            'sasl.username': KAFKA_API_KEY,
            'sasl.password': KAFKA_API_SECRET,
            'client.id': 'databricks-producer',
            'linger.ms': 100 
        })

    def delivery_report(self, err, msg):
        if err is not None:
            logger.error(f"Gửi lỗi: {err}")

    def on_message(self, ws, message):
        try:
            data = json.loads(message)
            kline = data['k']
            
            # Lọc lại payload, loại bỏ các trường rác của sàn
            payload = {
                "event_time": data['E'],
                "symbol": data['s'],
                "start_time": kline['t'],
                "open": float(kline['o']),
                "close": float(kline['c']),
                "high": float(kline['h']),
                "low": float(kline['l']),
                "volume": float(kline['v']),
                "is_closed": kline['x']
            }
            
            self.producer.produce(
                topic=self.topic,
                key=payload['symbol'],
                value=json.dumps(payload).encode('utf-8'),
                callback=self.delivery_report
            )
            self.producer.poll(0)
            
            if payload['is_closed']:
                logger.info(f"Đóng nến 1 phút: {payload['symbol']} | Giá: {payload['close']} | Vol: {payload['volume']}")

        except Exception as e:
            logger.error(f"Lỗi hệ thống: {e}")

    def on_error(self, ws, error):
        logger.error(f"Lỗi WebSocket: {error}")

    def on_close(self, ws, close_status_code, close_msg):
        logger.warning("Đứt kết nối. Đang đẩy phần dữ liệu còn lại lên Kafka...")
        self.producer.flush()

    def start(self):
        logger.info(f"🚀 Bắt đầu thu thập dữ liệu {self.symbol.upper()}...")
        while True:
            ws = websocket.WebSocketApp(
                self.ws_url,
                on_message=self.on_message,
                on_error=self.on_error,
                on_close=self.on_close
            )
            ws.run_forever(ping_interval=60, ping_timeout=10)
            logger.info("⏳ Thử kết nối lại sau 5 giây...")
            time.sleep(5)

# Khởi chạy Producer
if __name__ == "__main__":
    app = DatabricksBinanceProducer(symbol="btcusdt", interval="1m")
    try:
        app.start()
    except KeyboardInterrupt:
        logger.info("🛑 Dừng an toàn. Đang dọn dẹp hàng đợi Kafka...")
        app.producer.flush()
        logger.info("✅ Dừng thành công!")

In [0]:
df = spark.read.text("/Volumes/workspace/bronze/binance_raw/part-00000-b0c22da9-7c7d-41f2-8974-ed3351f7e6c4-c000.txt")
df_json = df.selectExpr("CAST(value AS STRING) as json_payload") \
    .selectExpr("from_json(json_payload, 'event_time LONG, symbol STRING, start_time LONG, open DOUBLE, close DOUBLE, high DOUBLE, low DOUBLE, volume DOUBLE, is_closed BOOLEAN') as data") \
    .select("data.*")
display(df_json)

In [0]:
df = spark.read.text("/Volumes/workspace/bronze/binance_raw/part-00000-1ad004f5-173c-4fc5-826c-1161ed6858c3-c000.txt")
df_json = df.selectExpr("CAST(value AS STRING) as json_payload") \
    .selectExpr("from_json(json_payload, 'event_time LONG, symbol STRING, start_time LONG, open DOUBLE, close DOUBLE, high DOUBLE, low DOUBLE, volume DOUBLE, is_closed BOOLEAN') as data") \
    .select("data.*")
display(df_json)

In [0]:
df = spark.read.text("/Volumes/workspace/bronze/binance_raw/part-00001-c964dee1-edb9-4014-8ab0-aa0daf2a9fe2-c000.txt")
df_json = df.selectExpr("CAST(value AS STRING) as json_payload") \
    .selectExpr("from_json(json_payload, 'event_time LONG, symbol STRING, start_time LONG, open DOUBLE, close DOUBLE, high DOUBLE, low DOUBLE, volume DOUBLE, is_closed BOOLEAN') as data") \
    .select("data.*")
display(df_json)